# Tattersalls Autumn Horses in Training Sale — Feature Engineering

**Notebook 03 of 3** | *TFM — Predictive Modelling of the Tattersalls Horses in Training Market*

This notebook implements a two-stage feature engineering pipeline:

- **Stage 1 — Classification**: predict whether a lot will sell to a third party.
- **Stage 2 — Regression**: predict `log(price_gns)` for lots that sell.

Features are derived from temporal structure, macroeconomic variables, pedigree signals,
entity target encoding, and interaction terms. A final permutation-importance audit
drives feature selection before export.

| § | Topic |
|---|-------|
| 0 | Setup & Imports |
| 0.5 | EDA → FE Decision Map |
| 1 | Quality Gates |
| 2 | Modelling Universe |
| 3 | Temporal & Macro Features |
| 4 | Entity Target Encoding |
| 5 | Interaction Features |
| 6 | Textual & Composite Features |
| 7 | Feature Selection (candidates, variance, VIF, permutation importance) |
| 8 | Diagnostic-driven Feature Reduction |
| &nbsp;&nbsp;&nbsp;8.5 | Cumulative importance concentration |
| &nbsp;&nbsp;&nbsp;8.6 | Feature redundancy diagnostic |
| &nbsp;&nbsp;&nbsp;8.7 | Final feature selection (hardcoded lists) |
| 9 | Final Datasets |
| 10 | Validation & Smoke Tests |
| 11 | Handoff to Modelling |

## 0. Setup & Imports

In [1]:
import sys
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.metrics import roc_auc_score, mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings('ignore')
PROJECT_ROOT = Path('').resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.data_prep import (
    extract_country_suffix, strip_country_suffix,
    normalize_root_entity, bootstrap_ci, PREMIUM_CUTOFF,
    get_macro_data
)
from src.constants import TRAIN_MAX_YEAR, VAL_MAX_YEAR, VAL_MIN_YEAR, TEST_MIN_YEAR
from src.sensors import temporal_split_validator, encoding_leakage_check, universe_consistency_check

RANDOM_STATE = 42
VIF_THRESHOLD = 10
M_ESTIMATE_M = 10
MIN_ENCODING_COUNT = 5

DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'

print('Setup OK')
print(f'PREMIUM_CUTOFF: {PREMIUM_CUTOFF:,} gns')
print(f'Train: ≤{TRAIN_MAX_YEAR}  |  Test: ≥{TEST_MIN_YEAR}')

Setup OK
PREMIUM_CUTOFF: 150,000 gns
Train: ≤2019  |  Test: ≥2022


In [2]:
df = pd.read_csv('..' / PROCESSED_DIR / 'autumn_horses_modeling_ready.csv')
enriched = pd.read_csv('..'/ PROCESSED_DIR / 'autumn_horses_eda_enriched.csv')

print(f'modeling_ready : {df.shape}')
print(f'eda_enriched   : {enriched.shape}')


modeling_ready : (26076, 32)
eda_enriched   : (26076, 64)


## 0.5. EDA → Feature Engineering: Decision Map

Each analytical conclusion from `02_EDA_Analysis` maps to an explicit implementation decision in this notebook:

| EDA Conclusion | Implementation | Section |
|:---|:---|:---:|
| **Hurdle model** (two stages): classification + conditional regression | Two-stage pipeline: `sold_to_third_party` → `log_price_gns` | Header |
| **Noise — Colour**: no causal link to performance | Excluded from `features_reg` via `EDA_REG_EXCLUDE`; removed by VIF in clf | §8.4 |
| **Noise — Foaled Month/Quarter**: marginal effect <5%, captured by `age_at_sale` | Excluded from `features_reg` via `EDA_REG_EXCLUDE` | §8.4 |
| **Noise — Sire-Dam Combo**: 90%+ singletons, critical leakage risk | Excluded from regression (`EDA_REG_EXCLUDE`); retained in clf as a novelty signal | §8.4 |
| **Noise — Grandsire, Sale Name**: redundant with sire / sale_year | Excluded in `ID_COLS` | §8 |
| **Noise — Dam Entity**: extreme cardinality, low repetition frequency | `dam_entity` in `ID_COLS`; signal captured by `dam_target_enc` (m=20, more permissive) | §5 |
| **day_normalized**: Day 1–2 price premium stable post-Brexit | `day`, `day_normalized`, `intraday_position`, `is_prime_time` | §3 |
| **Intraday prime time**: top-5% prices concentrated in central intraday bands | `is_prime_time` (60–80% intraday position) + continuous `intraday_position` | §3 |
| **Sires**: differentiated risk/return groups (Dubawi as benchmark) | `sire_target_enc` (m-estimate expanding), `sire_sale_rate_enc`, `sire_frequency` | §4–5 |
| **Consignors**: vendor reputation predicts price | `consignor_target_enc`, `consignor_volume`, `consignor_price_tier`, `consignor_sale_rate_enc` | §5–7 |
| **Damsires**: maternal genetic influence | `damsire_target_enc`, `damsire_count`, `damsire_sale_rate_enc` | §4–5 |
| **sex**: premium for reproductive potential (C ≫ F > G) | `sex_code`, dummies `sex_C/F/G/H/M`, interactions `sex_sire`, `sex_day` | §6–7 |
| **age_at_sale**: maturity and immediate competitive potential | Included in `features_reg`; dropped from clf by permutation importance (expected: low variance in this sale) | §8 |
| **Macro context**: currency, interest rates, market sentiment | `gbp_eur_rate`, `boe_base_rate`, `year_*_prior` with expanding LOO (no leakage) | §3 |
| **Anti-leakage target encoding** | M-estimate with expanding time window: only data prior to the observation year | §5 |


## 1. Quality Gates

In [3]:
# 1.1 Duplicados en clave compuesta
key_cols = ['sale_year', 'day', 'lot', 'horse_name_clean']
dups = df.duplicated(subset=key_cols, keep=False)
print(f'Duplicados en clave compuesta: {dups.sum()}')
if dups.sum() > 0:
    df = df.drop_duplicates(subset=key_cols, keep='first')
    print(f'  → Deduplicados. Shape: {df.shape}')

Duplicados en clave compuesta: 0


In [4]:
# 1.2 Logical inconsistencies
sold_no_price_or_free = (df['sold_to_third_party'] == True) & (df['price_gns'].isna() | (df['price_gns'] == 0))
print(f'Sold with no price or free: {sold_no_price_or_free.sum()}')

price_not_sold = (df['price_gns'] > 0) & (df['sold_to_third_party'] == False) & (df['vendor_buyback'] == False)
print(f'Positive price but not sold or buyback: {price_not_sold.sum()}')

age_anomaly = ~df['lot_withdrawn'] & ((df['age_at_sale'] < 1) | (df['age_at_sale'] > 7))
print(f'Anomalous age (<1 or >7) in non-withdrawn: {age_anomaly.sum()}')

Sold with no price or free: 0
Positive price but not sold or buyback: 781
Anomalous age (<1 or >7) in non-withdrawn: 40


In [5]:
# 1.3 Impossible values
neg_price = (df['price_gns'] < 0).sum()
print(f'Negative prices: {neg_price}')
print(f'\nNaN summary by critical column:')
critical = ['sex', 'sire_entity', 'dam_entity', 'damsire_entity', 'consignor_model', 'age_at_sale']
print(df[critical].isna().sum())

Negative prices: 0

NaN summary by critical column:
sex                22
sire_entity        27
dam_entity         27
damsire_entity     52
consignor_model    27
age_at_sale        22
dtype: int64


In [6]:
# I will exclude these NaNs in these critical features
print(f'Dropping rows with NaNs in critical features{df.shape} → ', end='')
df = df.dropna(subset=critical)
print(f'After dropping NaNs in critical features: {df.shape}')

Dropping rows with NaNs in critical features(26076, 32) → After dropping NaNs in critical features: (26019, 32)


## 2. Modelling Universe — Filtering & Scope

In [7]:
print(f'Total records: {len(df):,}')

# 2.1 Exclude Withdrawn
df = df[df['lot_withdrawn'] == False].copy()
print(f'After excluding withdrawn: {len(df):,}')

# 2.2 Exclude sex = R (Rig, 1 record)
df = df[df['sex'] != 'R'].copy()
print(f'After excluding R: {len(df):,}')

# 2.3 Exclude non-null 'Covered by' (mares in foal, n=6)
if 'stallion' in df.columns:
    covered = df['stallion'].notna() & (df['stallion'].str.strip() != '')
    df = df[~covered].copy()
    print(f'After excluding Covered by: {len(df):,}')
elif 'stallion' in enriched.columns:
    covered_idx = enriched[enriched['stallion'].notna() & (enriched['stallion'].astype(str).str.strip() != '')].index
    df = df[~df.index.isin(covered_idx)].copy()
    print(f'After excluding Covered by (via enriched): {len(df):,}')

# 2.4 Exclude stabling column (98% missing — remove column, not records)
if 'stabling' in df.columns:
    df = df.drop(columns=['stabling'])
    print('Column stabling removed')

# 2.5 Stable lot identifier for cross-dataset audit joins
_uid_cols = ['sale_name', 'sale_year', 'day', 'lot']
missing_uid = [c for c in _uid_cols if c not in df.columns]
assert not missing_uid, f'Missing UID columns: {missing_uid}'
df['lot_uid'] = (
    df['sale_name'].astype(str).str.strip() + '__' +
    df['sale_year'].astype(str) + '__d' + df['day'].astype(str) + '__l' + df['lot'].astype(str)
)
assert df['lot_uid'].is_unique, 'lot_uid must be unique after modelling-universe filters'
print(f'lot_uid created: {df["lot_uid"].nunique():,} unique lots')

# 2.6 Compute log_price_gns for vendor buybacks
# Vendor buybacks have a recorded price but log_price_gns
# was only computed for sold_to_third_party in 01_Data_Preparation.
vb_mask = (df['vendor_buyback'] == True) & df['price_gns'].gt(0)
df.loc[vb_mask, 'log_price_gns'] = np.log(df.loc[vb_mask, 'price_gns'])
print(f'log_price_gns now available for {vb_mask.sum()} vendor buyback lots')
print(f'  Range: [{df.loc[vb_mask, "log_price_gns"].min():.2f}, {df.loc[vb_mask, "log_price_gns"].max():.2f}]')

# Summary of price coverage by outcome
for outcome in ['sold_to_third_party', 'vendor_buyback', 'not_sold']:
    mask = df['sale_outcome'] == outcome
    n = mask.sum()
    has_log = df.loc[mask, 'log_price_gns'].notna().sum()
    print(f'  {outcome:25s}: {n:5d} rows, {has_log:5d} with log_price_gns')

Total records: 26,019
After excluding withdrawn: 18,972
After excluding R: 18,971
After excluding Covered by (via enriched): 18,966
lot_uid created: 18,966 unique lots
log_price_gns now available for 1379 vendor buyback lots
  Range: [6.68, 13.76]
  sold_to_third_party      : 16508 rows, 16508 with log_price_gns
  vendor_buyback           :  1379 rows,  1379 with log_price_gns
  not_sold                 :  1079 rows,     0 with log_price_gns


## 3. Temporal & Macroeconomic Features

In [8]:
macro = get_macro_data()
df = df.merge(macro, on='sale_year', how='left')
print(f'After merging macro data: {df.shape}')

After merging macro data: (18966, 35)


In [9]:
# 3.2 Annual market features (supply/demand/sentiment)
# Computed with LOO expanding window to avoid leakage.
# IMPORTANT: all market aggregates here are computed only on sold_to_third_party lots.
df = df.sort_values('sale_year').reset_index(drop=True)

year_stats = []
for yr in sorted(df['sale_year'].unique()):
    prior_sold = df[(df['sale_year'] < yr) & (df['sold_to_third_party'] == True)]
    if len(prior_sold) == 0:
        year_stats.append({'sale_year': yr, 'year_supply_prior': np.nan, 'year_demand_prior': np.nan,
                           'year_sale_rate_prior': np.nan, 'year_median_price_prior': np.nan})
        continue
    # Use the immediately prior year when available; otherwise fall back to all prior sold lots.
    prev_yr_sold = df[(df['sale_year'] == yr - 1) & (df['sold_to_third_party'] == True)]
    if len(prev_yr_sold) == 0:
        prev_yr_sold = prior_sold
    supply = len(prev_yr_sold)
    demand = prev_yr_sold['sold_to_third_party'].sum()
    sale_rate = demand / supply if supply > 0 else np.nan
    med_price = prev_yr_sold['price_gns'].median()
    year_stats.append({'sale_year': yr, 'year_supply_prior': supply, 'year_demand_prior': demand,
                       'year_sale_rate_prior': sale_rate, 'year_median_price_prior': med_price})

year_stats_df = pd.DataFrame(year_stats)
df = df.merge(year_stats_df, on='sale_year', how='left')
print('Year-level market features created:')
print(df[['sale_year','year_supply_prior','year_demand_prior','year_sale_rate_prior','year_median_price_prior']].drop_duplicates().sort_values('sale_year').to_string(index=False))

# Log-transform of year_median: additive relation with log_price_gns
# The tree learns: log_price ≈ log(year_median) + horse_relative_premium
# Without this, the model sees year_median in GNS and log_price in log — non-linear relation hard to learn
df['log_year_median_price_prior'] = np.log1p(df['year_median_price_prior'].fillna(df['year_median_price_prior'].median()))
print(f'log_year_median_price_prior: range [{df["log_year_median_price_prior"].min():.3f}, {df["log_year_median_price_prior"].max():.3f}]')

Year-level market features created:
 sale_year  year_supply_prior  year_demand_prior  year_sale_rate_prior  year_median_price_prior
      2009                NaN                NaN                   NaN                      NaN
      2010             903.00             903.00                  1.00                 9,000.00
      2011             865.00             865.00                  1.00                 9,000.00
      2012             848.00             848.00                  1.00                 9,000.00
      2013             909.00             909.00                  1.00                11,000.00
      2014             894.00             894.00                  1.00                10,000.00
      2015             922.00             922.00                  1.00                13,000.00
      2016           1,038.00           1,038.00                  1.00                10,000.00
      2017             948.00             948.00                  1.00                13,750.00
    

In [10]:
# 3.3 Intra-day features
# Convertir columnas que pueden llegar como ArrowStringArray a numérico
df['lot'] = pd.to_numeric(df['lot'], errors='coerce')
df['day'] = pd.to_numeric(df['day'], errors='coerce')

day_supply = df.groupby(['sale_year', 'day']).size().rename('day_supply').reset_index()
day_sales = df[df['sold_to_third_party'] == True].groupby(['sale_year', 'day']).size().rename('day_sold_count').reset_index()

df = df.merge(day_supply, on=['sale_year', 'day'], how='left')
df = df.merge(day_sales, on=['sale_year', 'day'], how='left')
df['day_sale_rate'] = df['day_sold_count'] / df['day_supply']

# Posición intra-día normalizada
lot_num = df['lot'].astype(float)
day_max_lot = df.groupby(['sale_year', 'day'])['lot'].transform('max').astype(float)
day_min_lot = df.groupby(['sale_year', 'day'])['lot'].transform('min').astype(float)
df['intraday_position'] = (lot_num - day_min_lot) / (day_max_lot - day_min_lot + 1e-9)

# Prime time: position between 60-80% of the day (EDA shows price peak)
df['is_prime_time'] = ((df['intraday_position'] >= 0.6) & (df['intraday_position'] <= 0.8) | ((df['intraday_position'] <= 0.4) & (df['intraday_position'] >= 0.2) & (df['day']==2))).astype(int)

print(f'day_supply range: {df["day_supply"].min()} – {df["day_supply"].max()}')
print(f'is_prime_time prevalence: {df["is_prime_time"].mean()*100:.1f}%')

day_supply range: 84 – 360
is_prime_time prevalence: 24.5%


## 4. Entity Target Encoding

In [11]:
def m_estimate_encoding(df_train, df_encode, entity_col, target_col, m=M_ESTIMATE_M, min_count=MIN_ENCODING_COUNT):
    """
    M-estimate target encoding with regularisation and temporal expanding window.
    For each row, uses only data prior to the observation year.
    Entities with <min_count occurrences are shrunk toward the global prior.
    """
    global_mean = df_train[target_col].mean()
    result = np.full(len(df_encode), global_mean)
    
    for idx in range(len(df_encode)):
        row = df_encode.iloc[idx]
        entity_val = row[entity_col]
        year_val = row['sale_year']
        
        # Solo datos anteriores (expanding window)
        prior_mask = (df_train[entity_col] == entity_val) & (df_train['sale_year'] < year_val)
        prior_data = df_train.loc[prior_mask, target_col].dropna()
        n = len(prior_data)
        
        if n == 0:
            result[idx] = global_mean
        else:
            entity_mean = prior_data.mean()
            # M-estimate: (n * entity_mean + m * global_mean) / (n + m)
            result[idx] = (n * entity_mean + m * global_mean) / (n + m)
    
    return result

In [12]:
# Price-observed subset for target encoding: sold to third party only.
# Vendor buybacks/RNAs are reserve-not-met non-transactions, so the final
# price signal is the realised third-party hammer price.
df_price = df[df['sold_to_third_party'] == True].copy()
print(f'df_price (sold-only encoding base): {len(df_price):,} rows')

# 4.1 Sire target encoding (price — for regression)
print('Computing sire_target_enc...')
df['sire_target_enc'] = m_estimate_encoding(df_price, df, 'sire_entity', 'log_price_gns')
print(f'  Done. Range: [{df["sire_target_enc"].min():.3f}, {df["sire_target_enc"].max():.3f}]')

# 4.2 Consignor target encoding (price)
print('Computing consignor_target_enc...')
df['consignor_target_enc'] = m_estimate_encoding(df_price, df, 'consignor_model', 'log_price_gns')
print(f'  Done. Range: [{df["consignor_target_enc"].min():.3f}, {df["consignor_target_enc"].max():.3f}]')

# 4.3 Damsire target encoding (price)
print('Computing damsire_target_enc...')
df['damsire_target_enc'] = m_estimate_encoding(df_price, df, 'damsire_entity', 'log_price_gns')
print(f'  Done. Range: [{df["damsire_target_enc"].min():.3f}, {df["damsire_target_enc"].max():.3f}]')

# 4.4 Dam target encoding (m=20: balance between high cardinality / signal preservation)
# m=50 collapsed values to near-global-mean → varianza<0.001 → eliminada por VarianceThreshold
print('Computing dam_target_enc (m=20)...')
df['dam_target_enc'] = m_estimate_encoding(df_price, df, 'dam_entity', 'log_price_gns', m=20)
print(f'  Done. Range: [{df["dam_target_enc"].min():.3f}, {df["dam_target_enc"].max():.3f}]')

# 4.5 Sale-rate encodings (for classification) — target: sold_to_third_party
# Use the full df (not only sold) to capture P(sold | entity)
df['sold_int'] = df['sold_to_third_party'].astype(int)

print('\nComputing sale-rate encodings (classification)...')
df['sire_sale_rate_enc'] = m_estimate_encoding(df, df, 'sire_entity', 'sold_int', m=10)
print(f'  sire_sale_rate_enc: [{df["sire_sale_rate_enc"].min():.3f}, {df["sire_sale_rate_enc"].max():.3f}]')

df['consignor_sale_rate_enc'] = m_estimate_encoding(df, df, 'consignor_model', 'sold_int', m=10)
print(f'  consignor_sale_rate_enc: [{df["consignor_sale_rate_enc"].min():.3f}, {df["consignor_sale_rate_enc"].max():.3f}]')

df['damsire_sale_rate_enc'] = m_estimate_encoding(df, df, 'damsire_entity', 'sold_int', m=10)
print(f'  damsire_sale_rate_enc: [{df["damsire_sale_rate_enc"].min():.3f}, {df["damsire_sale_rate_enc"].max():.3f}]')

df = df.drop(columns=['sold_int'])

print('\nEncoding summary (nulls):')
for col in ['sire_target_enc', 'consignor_target_enc', 'damsire_target_enc', 'dam_target_enc',
                        'sire_sale_rate_enc', 'consignor_sale_rate_enc', 'damsire_sale_rate_enc']:
        print(f'  {col}: {df[col].isna().sum()} null')

df_price (sold-only encoding base): 16,508 rows
Computing sire_target_enc...
  Done. Range: [8.625, 10.151]
Computing consignor_target_enc...
  Done. Range: [8.090, 10.789]
Computing damsire_target_enc...
  Done. Range: [8.592, 10.038]
Computing dam_target_enc (m=20)...
  Done. Range: [9.108, 9.736]

Computing sale-rate encodings (classification)...
  sire_sale_rate_enc: [0.638, 0.980]
  consignor_sale_rate_enc: [0.507, 0.997]
  damsire_sale_rate_enc: [0.580, 0.976]

Encoding summary (nulls):
  sire_target_enc: 0 null
  consignor_target_enc: 0 null
  damsire_target_enc: 0 null
  dam_target_enc: 0 null
  sire_sale_rate_enc: 0 null
  consignor_sale_rate_enc: 0 null
  damsire_sale_rate_enc: 0 null


In [13]:
# 4.5 Country effects — fixed: use M-estimate with expanding window (was global groupby → leakage)
# df_price (sold-only) already defined in §4.1 — used as encoding base.
df['horse_name_country'] = df['horse_name_country'].fillna('UNKNOWN')
df['country_target_enc'] = m_estimate_encoding(df_price, df, 'horse_name_country', 'log_price_gns')
print(f'country_target_enc: {df["country_target_enc"].nunique()} unique values')

# df_sold needed by cells in §6 (consignor specialization, price tier)
df_sold = df[df['sold_to_third_party'] == True].copy()

# 4.6 Entity frequency — fixed: expanding-window real (was cumcount — not deterministic within same year)
# For each row, count occurrences of the entity in ALL prior years.
df_sorted = df.sort_values(['sale_year', 'day', 'lot']).reset_index(drop=True)
sire_expanding = {}
consignor_expanding = {}
sire_freq_vals = np.zeros(len(df_sorted), dtype=int)
consignor_freq_vals = np.zeros(len(df_sorted), dtype=int)

for idx, row in df_sorted.iterrows():
    sire = row['sire_entity']
    cons = row['consignor_model']
    sire_freq_vals[idx] = sire_expanding.get(sire, 0)
    consignor_freq_vals[idx] = consignor_expanding.get(cons, 0)
    sire_expanding[sire] = sire_expanding.get(sire, 0) + 1
    consignor_expanding[cons] = consignor_expanding.get(cons, 0) + 1

# Align back to original df index
df['sire_frequency'] = pd.Series(sire_freq_vals, index=df_sorted.index)
df['consignor_frequency'] = pd.Series(consignor_freq_vals, index=df_sorted.index)
print(f'sire_frequency: median={df["sire_frequency"].median():.0f}')
print(f'consignor_frequency: median={df["consignor_frequency"].median():.0f}')

country_target_enc: 98 unique values
sire_frequency: median=26
consignor_frequency: median=59


## 5. Interaction Features

In [14]:
# Ordinal encoding of sex for interactions
sex_code_map = {'C': 2, 'G': 1, 'F': 0, 'H': 1.5, 'M': -0.5}
df['sex_code'] = df['sex'].map(sex_code_map).fillna(0)

# 5.1 sex × day
df['sex_day_interaction'] = df['sex_code'] * df['day']

# 5.2 sire_quality × sale_year — fixed: normalise sale_year with TRAIN stats only (was global mean/std → leakage)
train_mean_year = df.loc[df['sale_year'] <= TRAIN_MAX_YEAR, 'sale_year'].mean()
train_std_year = df.loc[df['sale_year'] <= TRAIN_MAX_YEAR, 'sale_year'].std()
sale_year_norm = (df['sale_year'] - train_mean_year) / train_std_year
df['sire_quality_year'] = df['sire_target_enc'] * sale_year_norm

# 5.3 age × sex
df['age_sex_interaction'] = df['age_at_sale'] * df['sex_code']

# 5.4 consignor × day
df['consignor_day_interaction'] = df['consignor_target_enc'] * df['day']

# 5.5 sex × sire_quality
df['sex_sire_interaction'] = df['sex_code'] * df['sire_target_enc']

interaction_cols = ['sex_day_interaction', 'sire_quality_year', 'age_sex_interaction',
                    'consignor_day_interaction', 'sex_sire_interaction']
print('Interaction features created:')
print(df[interaction_cols].describe().round(3).to_string())

Interaction features created:
       sex_day_interaction  sire_quality_year  age_sex_interaction  consignor_day_interaction  sex_sire_interaction
count            18,966.00          18,966.00            18,966.00                  18,966.00             18,966.00
mean                  2.41               9.13                 3.07                      23.19                  9.46
std                   2.16              14.55                 2.21                      10.62                  6.78
min                  -2.50             -15.16                -3.50                       8.09                 -4.99
25%                   0.00              -3.27                 0.00                       9.85                  0.00
50%                   2.00               8.61                 3.00                      20.10                  9.38
75%                   4.00              21.13                 4.00                      29.18                 17.92
max                  10.00              34

## 6. Textual & Composite Features

In [15]:
# 6.1 Pedigree novelty
combo_freq = df['sire_dam_combo'].value_counts().rename('sire_dam_combo_freq')
df['sire_dam_combo_freq'] = df['sire_dam_combo'].map(combo_freq).fillna(1)
df['sire_dam_combo_log_freq'] = np.log1p(df['sire_dam_combo_freq'])
df['is_novel_cross'] = (df['sire_dam_combo_freq'] == 1).astype(int)

print(f'Novel crosses (freq=1): {df["is_novel_cross"].sum()} ({df["is_novel_cross"].mean()*100:.1f}%)')
print(f'sire_dam_combo_freq: median={df["sire_dam_combo_freq"].median():.0f}, max={df["sire_dam_combo_freq"].max()}')

Novel crosses (freq=1): 16067 (84.7%)
sire_dam_combo_freq: median=1, max=4


In [16]:
# 6.2 Name features
df['name_length'] = df['horse_name_clean'].fillna('').str.len()
df['name_word_count'] = df['horse_name_clean'].fillna('').str.split().str.len()
noble_prefixes = ['MR', 'MRS', 'LADY', 'KING', 'QUEEN', 'PRINCE', 'LORD', 'SIR', 'DUKE', 'COUNT']
df['name_has_noble_prefix'] = df['horse_name_clean'].fillna('').str.upper().str.split().str[0].isin(noble_prefixes).astype(int)

print(f'name_length: mean={df["name_length"].mean():.1f}')
print(f'name_word_count: dist={df["name_word_count"].value_counts().sort_index().to_dict()}')
print(f'noble_prefix: {df["name_has_noble_prefix"].sum()} ({df["name_has_noble_prefix"].mean()*100:.1f}%)')

name_length: mean=11.0
name_word_count: dist={1: 7222, 2: 9502, 3: 1890, 4: 271, 5: 41, 6: 30, 7: 9, 8: 1}
noble_prefix: 341 (1.8%)


In [17]:
# 6.3 Consignor specialization
consignor_sex_mode = df.groupby('consignor_model')['sex'].agg(
    lambda x: x.value_counts().index[0] if len(x) > 0 else 'C'
).rename('consignor_sex_mode')
df = df.merge(consignor_sex_mode.reset_index(), on='consignor_model', how='left')

consignor_volume = df.groupby('consignor_model').size().rename('consignor_volume')
df = df.merge(consignor_volume.reset_index(), on='consignor_model', how='left')

# Price tier of consignor (using train split to compute percentiles)
train_consignor_median = df_sold[df_sold['sale_year'] <= TRAIN_MAX_YEAR].groupby('consignor_model')['price_gns'].median()
p33, p66 = train_consignor_median.quantile([0.33, 0.66])
consignor_tier_map = train_consignor_median.apply(lambda x: 'budget' if x < p33 else ('premium' if x > p66 else 'mid'))
df['consignor_price_tier'] = df['consignor_model'].map(consignor_tier_map).fillna('mid')

print(f'consignor_sex_mode dist: {df["consignor_sex_mode"].value_counts().to_dict()}')
print(f'consignor_price_tier dist: {df["consignor_price_tier"].value_counts().to_dict()}')

consignor_sex_mode dist: {'G': 14648, 'C': 3115, 'F': 1183, 'M': 18, 'H': 2}
consignor_price_tier dist: {'premium': 8785, 'mid': 8383, 'budget': 1798}


In [18]:
# 6.4 One-hot encoding de variables categóricas
# sex: dummies (C, F, G as base; H and M separate)
sex_dummies = pd.get_dummies(df['sex'], prefix='sex', drop_first=False).astype(int)
df = pd.concat([df, sex_dummies], axis=1)

# colour: dummies (top categorías, resto en 'other')
top_colours = df['colour'].value_counts().head(7).index
df['colour_clean'] = df['colour'].where(df['colour'].isin(top_colours), other='OTHER')
colour_dummies = pd.get_dummies(df['colour_clean'], prefix='colour', drop_first=False).astype(int)
df = pd.concat([df, colour_dummies], axis=1)

# foaled_quarter dummy
if 'foaled_quarter' in df.columns:
    quarter_dummies = pd.get_dummies(df['foaled_quarter'], prefix='q', drop_first=True).astype(int)
    df = pd.concat([df, quarter_dummies], axis=1)

print(f'Shape tras dummies: {df.shape}')
print(f'Sex dummies: {list(sex_dummies.columns)}')
print(f'Colour dummies: {list(colour_dummies.columns)}')

Shape tras dummies: (18966, 87)
Sex dummies: ['sex_C', 'sex_F', 'sex_G', 'sex_H', 'sex_M']
Colour dummies: ['colour_B', 'colour_B/Br', 'colour_Br', 'colour_Ch', 'colour_Gr', 'colour_Gr/Ro', 'colour_OTHER', 'colour_Ro']


## 7. Feature Selection

In [19]:
# Define candidate feature set
LEAKAGE_COLS = ['price_euros', 'purchaser', 'sale_outcome', 'price_gns',
                'vendor_buyback', 'lot_not_sold', 'lot_withdrawn', 'price_real_gns']
ID_COLS = ['horse_name_clean', 'horse_name_country', 'sire_entity', 'sire_clean', 'dam_entity',
           'dam_clean', 'damsire_entity', 'damsire_clean', 'grandsire_clean', 'consignor_model',
           'consignor_label', 'consignor_family', 'sire_dam_combo', 'colour', 'colour_clean',
           'sex', 'sale_name', 'consignor_sex_mode', 'consignor_price_tier', 'date_foaled',
           'sire_clean_join', 'sire_career_stage']
TARGET_COLS = ['sold_to_third_party', 'log_price_gns']

exclude_cols = set(LEAKAGE_COLS + ID_COLS + TARGET_COLS)
feature_candidates = [c for c in df.columns if c not in exclude_cols and df[c].dtype in [np.float64, np.int64, float, int, np.float32, np.int32]]

print(f'Feature candidates: {len(feature_candidates)}')
print(feature_candidates)

Feature candidates: 58
['sale_year', 'day', 'lot', 'birth_year', 'age_at_sale', 'foaled_month', 'foaled_quarter', 'gbp_eur_rate', 'boe_base_rate', 'year_supply_prior', 'year_demand_prior', 'year_sale_rate_prior', 'year_median_price_prior', 'log_year_median_price_prior', 'day_supply', 'day_sold_count', 'day_sale_rate', 'intraday_position', 'is_prime_time', 'sire_target_enc', 'consignor_target_enc', 'damsire_target_enc', 'dam_target_enc', 'sire_sale_rate_enc', 'consignor_sale_rate_enc', 'damsire_sale_rate_enc', 'country_target_enc', 'sire_frequency', 'consignor_frequency', 'sex_code', 'sex_day_interaction', 'sire_quality_year', 'age_sex_interaction', 'consignor_day_interaction', 'sex_sire_interaction', 'sire_dam_combo_freq', 'sire_dam_combo_log_freq', 'is_novel_cross', 'name_length', 'name_word_count', 'name_has_noble_prefix', 'consignor_volume', 'sex_C', 'sex_F', 'sex_G', 'sex_H', 'sex_M', 'colour_B', 'colour_B/Br', 'colour_Br', 'colour_Ch', 'colour_Gr', 'colour_Gr/Ro', 'colour_OTHER', 

In [20]:
# 7.1 Near-zero variance threshold
X_candidates = df[feature_candidates].fillna(df[feature_candidates].median())
vt = VarianceThreshold(threshold=0.001)
vt.fit(X_candidates)

low_var_mask = ~vt.get_support()
low_var_cols = [c for c, m in zip(feature_candidates, low_var_mask) if m]
print(f'Low variance features removed: {len(low_var_cols)}')
if low_var_cols:
    print(f'  {low_var_cols}')

features_after_vt = [c for c in feature_candidates if c not in low_var_cols]
print(f'Features after variance filter: {len(features_after_vt)}')

Low variance features removed: 3
  ['year_sale_rate_prior', 'q_3.0', 'q_4.0']
Features after variance filter: 55


In [21]:
# 7.2 VIF — omitted for tree-based models
# VIF with threshold=10 makes sense for linear regression (unstable coefficients with correlated features).
# RF and GBT do not suffer from multicollinearity: they can use correlated features without losing stability.
# Applying VIF here would remove sire_target_enc, consignor_target_enc, day, year_sale_rate_prior —
# exactly the signals the EDA identifies as most predictive. Selection is delegated
# to permutation importance (Section 8.3), which measures real impact on the final model.
features_vif = list(features_after_vt)
removed_vif = []
print(f'VIF step omitted (tree models do not suffer from multicollinearity).')
print(f'Features entering permutation importance: {len(features_vif)}')

VIF step omitted (tree models do not suffer from multicollinearity).
Features entering permutation importance: 55


In [22]:
# 7.3 Permutation importance (pre-model)
train_mask = df['sale_year'] <= TRAIN_MAX_YEAR
test_mask = df['sale_year'] >= TEST_MIN_YEAR

X_train = df.loc[train_mask, features_vif].fillna(df[features_vif].median())
y_train_clf = df.loc[train_mask, 'sold_to_third_party'].astype(int)
y_train_reg = df.loc[train_mask & (df['sold_to_third_party'] == True), 'log_price_gns']

X_test = df.loc[test_mask, features_vif].fillna(df[features_vif].median())
y_test_clf = df.loc[test_mask, 'sold_to_third_party'].astype(int)

# class_weight='balanced' — sin esto el RF colapsa a predecir clase mayoritaria (87%)
# y la permutation importance resulta 0.0 para todas las features
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=8, n_jobs=-1,
                                class_weight='balanced', random_state=RANDOM_STATE)
rf_clf.fit(X_train, y_train_clf)

perm_imp = permutation_importance(rf_clf, X_test, y_test_clf, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
perm_df = pd.DataFrame({
    'feature': features_vif,
    'importance_mean': perm_imp.importances_mean,
    'importance_std': perm_imp.importances_std
}).sort_values('importance_mean', ascending=False)

print('Top 20 features by permutation importance:')
print(perm_df.head(20).to_string(index=False))

Top 20 features by permutation importance:
              feature  importance_mean  importance_std
  consignor_frequency             0.00            0.00
       sire_frequency             0.00            0.00
           day_supply             0.00            0.00
       dam_target_enc             0.00            0.00
   sire_sale_rate_enc             0.00            0.00
          age_at_sale             0.00            0.00
   country_target_enc             0.00            0.00
                q_2.0             0.00            0.00
       foaled_quarter             0.00            0.00
            colour_Gr             0.00            0.00
          colour_B/Br             0.00            0.00
         foaled_month             0.00            0.00
                  lot             0.00            0.00
            colour_Br             0.00            0.00
name_has_noble_prefix             0.00            0.00
                sex_M             0.00            0.00
           birth_year 

> **Note — near-zero permutation importances in classification:** With an 87% majority class (sold),
> the Random Forest with `class_weight='balanced'` produces relatively stable probability predictions
> under single-feature permutation on the OOT set, yielding importances close to zero.
> This **does not mean the features are uninformative**: the smoke test with HistGradientBoosting
> (§10.3) confirms AUC-ROC OOT = 0.624.
>
> `features_clf` selection uses a conservative threshold `imp < −0.002` (remove only features
> that actively hurt performance) plus explicit protection of EDA market signals (`CLF_EXTRAS`).
> The final modelling HGB will apply its own implicit regularisation.
> For `features_reg`, permutation importance yields a clear signal (§8.4), as the regressor
> is trained on the sold subset — more homogeneous and without the extreme class imbalance
> present in the classification stage.

In [23]:
# 8.3b — features_clf: based on permutation importance of the classifier (Section 8.3)
# features_reg is now defined in §8.7 via reg_signal_selected (after redundancy check).
# The placeholder `features_reg = features_final` has been removed — it was overwritten in §8.4 anyway.
# Variables features_clf / features_reg below are kept for backward compat (cells 10.1, 10.2).

noise_features = perm_df[perm_df['importance_mean'] < -0.002]['feature'].tolist()
print(f'Features removed as noise for classification (imp < -0.002): {noise_features}')

features_final = [f for f in features_vif if f not in noise_features]
print(f'\nFinal feature count for classification: {len(features_final)}')

# Classification: protect temporal market signals (EDA Section 3)
CLF_EXTRAS = ['day', 'day_normalized', 'year_sale_rate_prior', 'day_sale_rate',
               'year_demand_prior', 'year_supply_prior']
clf_extras = [f for f in CLF_EXTRAS if f in df.columns and f not in features_final]
features_clf = features_final + clf_extras
print(f'features_clf: {len(features_clf)}  (+{len(clf_extras)} protected: {clf_extras})')
print('features_reg: will be defined in §8.7 (regression perm importance + redundancy check)')

Features removed as noise for classification (imp < -0.002): ['sex_code', 'is_novel_cross', 'consignor_day_interaction', 'day_sale_rate', 'sex_day_interaction', 'sex_sire_interaction', 'sire_dam_combo_freq', 'consignor_sale_rate_enc', 'consignor_target_enc', 'consignor_volume']

Final feature count for classification: 45
features_clf: 47  (+2 protected: ['year_sale_rate_prior', 'day_sale_rate'])
features_reg: will be defined in §8.7 (regression perm importance + redundancy check)


In [24]:
# 8.4 Permutation importance for REGRESSION (independent of classifier)
# Correct hurdle model: each stage selects its own features.
# Classification perm importance favours sale signals (day_sale_rate, market signals).
# Regression perm importance favours price signals (sire_target_enc, consignor quality).
# Using the same selection for both tasks is a methodological flaw.

train_sold_mask = (df['sale_year'] <= TRAIN_MAX_YEAR) & (df['sold_to_third_party'] == True)
test_sold_mask  = (df['sale_year'] >= TEST_MIN_YEAR)  & (df['sold_to_third_party'] == True)

# Candidates: full features_vif — without filtering by classification importance
reg_candidates = [f for f in features_vif if f in df.columns]

X_tr_reg_sel = df.loc[train_sold_mask, reg_candidates].fillna(df[reg_candidates].median())
y_tr_reg_sel  = df.loc[train_sold_mask, 'log_price_gns']
X_te_reg_sel  = df.loc[test_sold_mask,  reg_candidates].fillna(df[reg_candidates].median())
y_te_reg_sel  = df.loc[test_sold_mask,  'log_price_gns']

rf_reg_sel = RandomForestRegressor(n_estimators=100, max_depth=10, n_jobs=-1, random_state=RANDOM_STATE)
rf_reg_sel.fit(X_tr_reg_sel, y_tr_reg_sel)

perm_reg = permutation_importance(
    rf_reg_sel, X_te_reg_sel, y_te_reg_sel,
    n_repeats=10, scoring='neg_root_mean_squared_error',
    random_state=RANDOM_STATE, n_jobs=-1
)
perm_reg_df = pd.DataFrame({
    'feature': reg_candidates,
    'importance_mean': perm_reg.importances_mean,
    'importance_std': perm_reg.importances_std
}).sort_values('importance_mean', ascending=False)

print('Top 20 features by importance in REGRESSION:')
print(perm_reg_df.head(20).to_string(index=False))

# features_reg: features with importance >= -0.002 in the price task
reg_noise = perm_reg_df[perm_reg_df['importance_mean'] < -0.002]['feature'].tolist()
features_reg = [f for f in reg_candidates if f not in reg_noise]

# Ensure that price encodings from the EDA are never removed
REG_MUST_HAVE = ['sire_target_enc', 'consignor_target_enc', 'damsire_target_enc',
                  'day', 'day_normalized', 'age_at_sale', 'log_year_median_price_prior']
for f in REG_MUST_HAVE:
    if f in df.columns and f not in features_reg:
        features_reg.append(f)

print(f'\nfeatures_reg final: {len(features_reg)} features')
print(f'Noise removed: {reg_noise}')
print(f'Must-have added: {[f for f in REG_MUST_HAVE if f in df.columns]}')
# ── EDA §6 Noise Audit: explicit exclusion of features flagged as noise ──────────
EDA_REG_EXCLUDE = [
    # Colour: no causal relation with performance; noisy proxy for sire (EDA §6)
    'colour_B', 'colour_B/Br', 'colour_Br', 'colour_Ch',
    'colour_Gr', 'colour_Gr/Ro', 'colour_OTHER', 'colour_Ro',
    # Foaling: no material price variation across months/quarters (EDA §6, §7)
    'foaled_month', 'foaled_quarter', 'q_2.0',
    # Sire-Dam combo: extreme cardinality → critical leakage risk (EDA §6)
    'sire_dam_combo_freq', 'sire_dam_combo_log_freq', 'is_novel_cross',
    # Name features: speculative, no causal relation with price
    'name_length', 'name_word_count', 'name_has_noble_prefix',
]
_before = len(features_reg)
features_reg = [f for f in features_reg if f not in EDA_REG_EXCLUDE]
print(f'features_reg after EDA noise audit exclusion: {len(features_reg)} features  (-{_before - len(features_reg)} noise)')

Top 20 features by importance in REGRESSION:
                  feature  importance_mean  importance_std
consignor_day_interaction             0.09            0.01
        intraday_position             0.08            0.00
                      day             0.02            0.00
     sex_sire_interaction             0.01            0.00
      sex_day_interaction             0.01            0.00
      age_sex_interaction             0.01            0.00
            day_sale_rate             0.00            0.00
     consignor_target_enc             0.00            0.00
  consignor_sale_rate_enc             0.00            0.00
       sire_sale_rate_enc             0.00            0.00
          sire_target_enc             0.00            0.00
           sire_frequency             0.00            0.00
              age_at_sale             0.00            0.00
              name_length             0.00            0.00
       country_target_enc             0.00            0.00
           

In [25]:
def cumulative_importance_report(perm_df: pd.DataFrame, title: str, total_features: int) -> list:
    """Print cumulative importance and return features to retain (|imp| >= 0.0005)."""
    df = perm_df.copy()
    df['abs_imp'] = df['importance_mean'].abs()
    df = df.sort_values('abs_imp', ascending=False)
    df['cum_pct'] = df['abs_imp'].cumsum() / df['abs_imp'].sum() * 100
    
    print(f'=== {title} ===')
    print(f'Total features: {total_features}')
    for _, r in df.iterrows():
        marker = '' if r['abs_imp'] >= 0.0005 else ' ⬜ near-zero'
        print(f'  {r["feature"]:40s} Absolute Importance Percentage={r["abs_imp"]*100:+.2f}%  cum={r["cum_pct"]:5.1f}%{marker}')
    
    n_signal = (df['abs_imp'] >= 0.0005).sum()
    print(f'\nFeatures with |imp| >= 0.0005: {n_signal}/{total_features}')
    print(f'Top 5 capture: {df.head(5)["abs_imp"].sum() / df["abs_imp"].sum() * 100:.1f}%')
    print(f'Top 10 capture: {df.head(10)["abs_imp"].sum() / df["abs_imp"].sum() * 100:.1f}%')
    print(f'Top 15 capture: {df.head(15)["abs_imp"].sum() / df["abs_imp"].sum() * 100:.1f}%')
    print(f'Top 20 capture: {df.head(20)["abs_imp"].sum() / df["abs_imp"].sum() * 100:.1f}%')
    print(f"Top 30 capture: {df.head(30)['abs_imp'].sum() / df['abs_imp'].sum() * 100:.1f}%")
    
    return df[df['abs_imp'] >= 0.0005].sort_values('abs_imp', ascending=False)['feature'].tolist()

clf_signal_features = cumulative_importance_report(perm_df, 'Classification importance concentration', len(features_vif))
reg_signal_features = cumulative_importance_report(perm_reg_df, 'Regression importance concentration', len(reg_candidates))


=== Classification importance concentration ===
Total features: 55
  consignor_volume                         Absolute Importance Percentage=+1.38%  cum= 17.6%
  consignor_target_enc                     Absolute Importance Percentage=+1.32%  cum= 34.4%
  consignor_sale_rate_enc                  Absolute Importance Percentage=+0.90%  cum= 45.9%
  sire_dam_combo_freq                      Absolute Importance Percentage=+0.45%  cum= 51.6%
  sex_sire_interaction                     Absolute Importance Percentage=+0.41%  cum= 56.9%
  sex_day_interaction                      Absolute Importance Percentage=+0.37%  cum= 61.7%
  day_sale_rate                            Absolute Importance Percentage=+0.34%  cum= 66.1%
  consignor_day_interaction                Absolute Importance Percentage=+0.33%  cum= 70.3%
  is_novel_cross                           Absolute Importance Percentage=+0.30%  cum= 74.1%
  sex_code                                 Absolute Importance Percentage=+0.30%  cum= 78.0%
  c

Analyzing these results, I will select the top 5 in regression and top 17 in classification. The others are complete noise

In [26]:
reg_signal_selected = reg_signal_features[:5]  # Top 5 features by regression importance
print(f'\nTop regression features selected for final model: {reg_signal_selected}')
clf_signal_selected = clf_signal_features[:17]  # Top 17 features by classification importance
print(f'Top classification features selected for final model: {clf_signal_selected}')


Top regression features selected for final model: ['consignor_day_interaction', 'intraday_position', 'day', 'sex_sire_interaction', 'sex_day_interaction']
Top classification features selected for final model: ['consignor_volume', 'consignor_target_enc', 'consignor_sale_rate_enc', 'sire_dam_combo_freq', 'sex_sire_interaction', 'sex_day_interaction', 'day_sale_rate', 'consignor_day_interaction', 'is_novel_cross', 'sex_code', 'consignor_frequency', 'intraday_position', 'age_sex_interaction', 'day_sold_count', 'sire_dam_combo_log_freq', 'name_length', 'year_supply_prior']


In [27]:
# ── 8.6 Feature redundancy diagnostic ──
# Identifies pairs of features with near-perfect correlation.
# When r > 0.9, the weaker feature (lower |imp|) is redundant.

def redundancy_report(df_data: pd.DataFrame, candidates: list, imp_dict: dict, threshold: float = 0.8) -> set:
    """Find feature pairs with |r| > threshold; return set of redundant features."""
    present = [c for c in candidates if c in df_data.columns]
    corr = df_data[present].corr().abs()
    redundant = set()
    print(f'=== Redundancy analysis (|r| > {threshold}) ===')
    for i in range(len(present)):
        for j in range(i+1, len(present)):
            if corr.iloc[i, j] > threshold:
                imp_i = abs(imp_dict.get(present[i], 0))
                imp_j = abs(imp_dict.get(present[j], 0))
                to_drop = present[j] if imp_i >= imp_j else present[i]
                redundant.add(to_drop)
                print(f'  {present[i]:35s} ↔ {present[j]:35s}  r={corr.iloc[i,j]:.3f}  → drop {to_drop}')
    if not redundant:
        print('  None found — features are reasonably orthogonal')
    return redundant

# Build importance dicts from permutation results
clf_imp = dict(zip(perm_df['feature'], perm_df['importance_mean']))
reg_imp = dict(zip(perm_reg_df['feature'], perm_reg_df['importance_mean']))

print("classification redundancy check:")
clf_redundant = redundancy_report(df, clf_signal_selected, clf_imp)
print("\n\nregression redundancy check:")
reg_redundant = redundancy_report(df[df['sold_to_third_party'] == True], reg_signal_selected, reg_imp)

classification redundancy check:
=== Redundancy analysis (|r| > 0.8) ===
  sire_dam_combo_freq                 ↔ is_novel_cross                       r=0.947  → drop is_novel_cross
  sire_dam_combo_freq                 ↔ sire_dam_combo_log_freq              r=0.996  → drop sire_dam_combo_log_freq
  sex_sire_interaction                ↔ sex_code                             r=0.999  → drop sex_code
  sex_sire_interaction                ↔ age_sex_interaction                  r=0.866  → drop age_sex_interaction
  is_novel_cross                      ↔ sire_dam_combo_log_freq              r=0.971  → drop sire_dam_combo_log_freq
  sex_code                            ↔ age_sex_interaction                  r=0.865  → drop age_sex_interaction


regression redundancy check:
=== Redundancy analysis (|r| > 0.8) ===
  consignor_day_interaction           ↔ day                                  r=0.997  → drop day


There are some redundancies in the selected features, so I will get the top 15 variables in each dataset 12 features in regression and 20 features in classification (more than 90% importance capture), excluding these

In [28]:
reg_signal_selected = [f for f in reg_signal_features if f not in reg_redundant][:12]
clf_signal_selected = [f for f in clf_signal_features if f not in clf_redundant][:20]
print(f'\nFinal regression features after redundancy check: {reg_signal_selected}')
print(f'Final classification features after redundancy check: {clf_signal_selected}')


Final regression features after redundancy check: ['consignor_day_interaction', 'intraday_position', 'sex_sire_interaction', 'sex_day_interaction', 'age_sex_interaction', 'day_sale_rate', 'consignor_target_enc', 'consignor_sale_rate_enc', 'lot', 'sire_sale_rate_enc', 'consignor_volume', 'sire_target_enc']
Final classification features after redundancy check: ['consignor_volume', 'consignor_target_enc', 'consignor_sale_rate_enc', 'sire_dam_combo_freq', 'sex_sire_interaction', 'sex_day_interaction', 'day_sale_rate', 'consignor_day_interaction', 'consignor_frequency', 'intraday_position', 'day_sold_count', 'name_length', 'year_supply_prior', 'sire_frequency', 'gbp_eur_rate', 'day_supply', 'dam_target_enc']


## 9. Final Datasets

Three datasets are exported for `04_Modeling`:

| Dataset | Universe | Rows (approx.) | Features | Purpose |
|---|---|---|---|---|
| `classification_ready` | All offered lots | ~18,966 | **20 selected** + meta | Stage 1 — train P(sold_to_third_party) |
| `regression_ready` | Sold to third party only | ~16,500 | **12 selected** + meta | Stage 2 — train log(price) regressor |
| `inference_universe` | All offered lots | ~18,966 | **~25 selected** + meta | Stage 2 — apply regressor to all lots |

Feature selection was data-driven: cumulative permutation importance (≥0.0005 threshold) + redundancy diagnostic (|r| > 0.8). See §8.5–8.7 for full methodology.

The separation of `regression_ready` (training) from `inference_universe` (prediction)
is intentional: the regressor trains on lots with observable market-clearing prices, then predicts
a counterfactual fair-market-value for every offered lot — including buybacks and not-sold lots.

**Note:** Vendor buybacks/RNAs (~1,382 lots) have a recorded reserve price but are NOT
market-clearing transactions. They are excluded from Stage 2 training and reserved for
audit/ablation analysis in `05_Model_Audit`.

In [29]:
# 9.1 Classification dataset (Stage 1)
# Reference columns for downstream notebooks
classification_ready = df[clf_signal_selected + ['sale_year','sold_to_third_party','lot_uid','vendor_buyback','lot_not_sold']].copy()

# Temporally-safe imputation: median computed only from PRIOR years
median_fill_cols = classification_ready.columns[classification_ready.isna().any()].tolist()
for col in median_fill_cols:
    for yr in sorted(classification_ready['sale_year'].unique()):
        mask = classification_ready['sale_year'] == yr
        prior_median = classification_ready.loc[classification_ready['sale_year'] < yr, col].median()
        if pd.isna(prior_median):
            prior_median = classification_ready[col].median()
        classification_ready.loc[mask, col] = classification_ready.loc[mask, col].fillna(prior_median)

print(f'classification_ready: {classification_ready.shape}')
print(f'  Total nulls remaining: {classification_ready.isna().sum().sum()}')
print(f'  Sale rate: {classification_ready["sold_to_third_party"].mean()*100:.1f}%')
print(f'  Train: {(classification_ready["sale_year"] <= TRAIN_MAX_YEAR).sum():,}')
print(f'  Test:  {(classification_ready["sale_year"] >= TEST_MIN_YEAR).sum():,}')

classification_ready: (18966, 22)
  Total nulls remaining: 0
  Sale rate: 87.0%
  Train: 12,097
  Test:  4,671


In [30]:
# 9.2 Stage 2 — Regression training dataset
# Training universe: sold_to_third_party only.
# Vendor buybacks/RNAs are reserve-not-met non-transactions, not realised market prices.
# They remain available in inference/audit datasets but are excluded from Stage 2 training.
regression_ready = df[df['sold_to_third_party'] == True][reg_signal_selected + ['sold_to_third_party', 'vendor_buyback', 'sale_year', 'log_price_gns','lot_uid']].dropna(subset=['log_price_gns']).copy()

# Temporal-safe imputation
median_fill_cols_reg = regression_ready.columns[regression_ready.isna().any()].tolist()
for col in median_fill_cols_reg:
    if col in ['lot_uid','log_price_gns', 'sold_to_third_party', 'vendor_buyback']:
        continue
    for yr in sorted(regression_ready['sale_year'].unique()):
        mask = regression_ready['sale_year'] == yr
        prior_median = regression_ready.loc[regression_ready['sale_year'] < yr, col].median()
        if pd.isna(prior_median):
            prior_median = regression_ready[col].median()
        regression_ready.loc[mask, col] = regression_ready.loc[mask, col].fillna(prior_median)

print(f'regression_ready: {regression_ready.shape}')
print(f'  sold_to_third_party : {regression_ready["sold_to_third_party"].sum():,}')
print(f'  vendor_buyback      : {regression_ready["vendor_buyback"].sum():,}')
print(f'  log_price_gns — mean={regression_ready["log_price_gns"].mean():.3f}, '
      f'std={regression_ready["log_price_gns"].std():.3f}')
print(f'  Train (≤{TRAIN_MAX_YEAR}): {(regression_ready["sale_year"] <= TRAIN_MAX_YEAR).sum():,}')
print(f'  OOT   (≥{TEST_MIN_YEAR}): {(regression_ready["sale_year"] >= TEST_MIN_YEAR).sum():,}')
print(f'  Total nulls: {regression_ready.isna().sum().sum()}')


regression_ready: (16508, 17)
  sold_to_third_party : 16,508
  vendor_buyback      : 0
  log_price_gns — mean=9.375, std=1.267
  Train (≤2019): 10,424
  OOT   (≥2022): 4,113
  Total nulls: 0


### 9.3 Inference Universe

A third dataset containing **all offered lots** (sold + buyback + not_sold) with the
full set of regression features. Used in `04_Modeling` to apply the trained Stage 2
regressor to every offered lot — including those that did not sell — to obtain a
counterfactual *fair market value* estimate for each horse.

In [31]:
# 9.3 inference_universe: all offered lots with regression features
# Purpose: apply the trained Stage 2 regressor to ALL offered lots in 04_Modeling.
# log_price_gns is present where observable (sold + buyback); NaN for not_sold → predicted.
# Uses the final selected feature lists from §8.7.
inf_feature_cols = list(dict.fromkeys(reg_signal_selected + clf_signal_selected))
inf_cols = inf_feature_cols + [col for col in
    ['lot_uid', 'log_price_gns', 'price_gns', 'sale_year',
     'sold_to_third_party', 'vendor_buyback', 'lot_not_sold']
    if col not in inf_feature_cols]

inference_universe = df[[c for c in inf_cols if c in df.columns]].copy()

# Temporal-safe imputation for features (not for targets)
median_fill_cols_inf = inference_universe.columns[inference_universe.isna().any()].tolist()
skip_inf = ['lot_uid', 'log_price_gns', 'price_gns', 'sold_to_third_party',
            'vendor_buyback', 'lot_not_sold']
for col in median_fill_cols_inf:
    if col in skip_inf:
        continue
    for yr in sorted(inference_universe['sale_year'].unique()):
        mask = inference_universe['sale_year'] == yr
        prior_median = inference_universe.loc[inference_universe['sale_year'] < yr, col].median()
        if pd.isna(prior_median):
            prior_median = inference_universe[col].median()
        inference_universe.loc[mask, col] = inference_universe.loc[mask, col].fillna(prior_median)

print(f'inference_universe: {inference_universe.shape}')
print(f'  sold_to_third_party : {inference_universe["sold_to_third_party"].sum():,}')
print(f'  vendor_buyback      : {inference_universe["vendor_buyback"].sum():,}')
print(f'  not_sold            : {inference_universe["lot_not_sold"].sum():,}')
print(f'  log_price_gns known : {inference_universe["log_price_gns"].notna().sum():,}  '
      f'(will be predicted for remaining {inference_universe["log_price_gns"].isna().sum():,})')

inference_universe: (18966, 28)
  sold_to_third_party : 16,508
  vendor_buyback      : 1,379
  not_sold            : 1,079
  log_price_gns known : 17,887  (will be predicted for remaining 1,079)


In [32]:
# ── Data integrity sensors ──────────────────────────────────────────────
# Validate temporal splits, target encoding anti-leakage, and universe consistency
# before exporting to 04_Modeling. These checks enforce the invariants in AGENTS.md §2.2.

# Temporal split validation (uses constants defined in src.constants)
train_split = df[df['sale_year'] <= TRAIN_MAX_YEAR]
val_split   = df[(df['sale_year'] >= VAL_MIN_YEAR) & (df['sale_year'] <= VAL_MAX_YEAR)]
oot_split   = df[df['sale_year'] >= TEST_MIN_YEAR]

temporal_split_validator(train_split, val_split, oot_split, year_col='sale_year')
print('temporal_split_validator: PASSED')

# Target encoding anti-leakage check (price encodings — includes M-estimate recomputation)
# Pass encoding_mask to match notebook 03's sold-only encoding base (df_price).
# Without this mask, the sensor's global_mean would include vendor buyback prices,
# producing a different M-estimate from the canonical feature engineering.
# The dam entity uses m=20 (high cardinality → stronger regularisation), passed
# as a 3-tuple (entity, encoded, pair_m) to override the default m=10.
entity_pairs_price = [
    ('sire_entity',     'sire_target_enc'),
    ('consignor_model', 'consignor_target_enc'),
    ('damsire_entity',  'damsire_target_enc'),
    ('dam_entity',      'dam_target_enc', 20),
]
encoding_leakage_check(df, entity_pairs_price, year_col='sale_year',
                       target_col='log_price_gns', m=M_ESTIMATE_M,
                       encoding_mask=df['sold_to_third_party'] == True)
print('encoding_leakage_check (price): PASSED')

# Target encoding anti-leakage check (sale-rate encodings — structural check only)
# The function skips recomputation for columns containing "sale_rate" in their name.
entity_pairs_sale = [
    ('sire_entity',     'sire_sale_rate_enc'),
    ('consignor_model', 'consignor_sale_rate_enc'),
    ('damsire_entity',  'damsire_sale_rate_enc'),
]
encoding_leakage_check(df, entity_pairs_sale, year_col='sale_year')
print('encoding_leakage_check (sale-rate): PASSED')

# Universe consistency: all regression_ready IDs must exist in inference_universe
universe_consistency_check(inference_universe, regression_ready, id_cols=['lot_uid'])
print('universe_consistency_check: PASSED')

print('\n── All data integrity sensors: PASSED ──')

temporal_split_validator: PASSED
encoding_leakage_check (price): PASSED
encoding_leakage_check (sale-rate): PASSED
universe_consistency_check: PASSED

── All data integrity sensors: PASSED ──


In [33]:
# 9.4 Export all three datasets
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

datasets = {
    'classification_ready': classification_ready,
    'regression_ready':     regression_ready,
    'inference_universe':   inference_universe,
}

for name, dset in datasets.items():
    dset.to_parquet(PROCESSED_DIR / f'{name}.parquet', index=False)

print('Exports:')
for name in datasets:
    for ext in ['csv', 'parquet']:
        p = PROCESSED_DIR / f'{name}.{ext}'
        print(f'  {name}.{ext:<10s}: {p.stat().st_size / 1024:>6.0f} KB')


Exports:
  classification_ready.csv       :   4431 KB
  classification_ready.parquet   :    592 KB
  regression_ready.csv       :   4008 KB
  regression_ready.parquet   :    558 KB
  inference_universe.csv       :   5760 KB
  inference_universe.parquet   :    752 KB


## 10. Validation & Smoke Tests

In [34]:
# 10.1 Leakage audit
leakage_check = ['price_euros', 'purchaser', 'sale_outcome', 'price_gns']
for col in leakage_check:
    assert col not in classification_ready.columns, f'LEAKAGE: {col} in classification_ready'
    assert col not in regression_ready.columns, f'LEAKAGE: {col} in regression_ready'
print('Leakage audit: PASSED')

# High correlation with target (leakage signal) — using selected features
corr_cols = [f for f in reg_signal_selected if f in regression_ready.columns]
corr_with_target = regression_ready[corr_cols].corrwith(regression_ready['log_price_gns']).abs()
suspicious = corr_with_target[corr_with_target > 0.95]
if len(suspicious) > 0:
    print(f'WARNING: Features with correlation >0.95 with target:')
    print(suspicious)
else:
    print(f'Maximum correlation with target: {corr_with_target.max():.3f} — OK')

Leakage audit: PASSED
Maximum correlation with target: 0.263 — OK


In [35]:
# 10.2 KS-test train vs test distribution — using selected regression features
train_reg = regression_ready[regression_ready['sale_year'] <= TRAIN_MAX_YEAR]
test_reg = regression_ready[regression_ready['sale_year'] >= TEST_MIN_YEAR]

ks_results = []
for feat in [f for f in reg_signal_selected if f in regression_ready.columns][:10]:
    t_vals = train_reg[feat].dropna()
    v_vals = test_reg[feat].dropna()
    if len(t_vals) > 10 and len(v_vals) > 10:
        ks_stat, ks_p = stats.ks_2samp(t_vals, v_vals)
        ks_results.append({'feature': feat, 'ks_stat': round(ks_stat, 3), 'p_value': round(ks_p, 4), 'drift': ks_p < 0.05})

ks_df = pd.DataFrame(ks_results)
drift_rate = ks_df['drift'].mean() * 100
print(f'KS-test (top 10 features): {drift_rate:.0f}% con drift temporal (p<0.05)')
if drift_rate > 20:
    print('WARNING: >20% con drift — revisar features temporales')
    print(ks_df[ks_df['drift']].to_string(index=False))

KS-test (top 10 features): 90% con drift temporal (p<0.05)
                  feature  ks_stat  p_value  drift
consignor_day_interaction     0.14     0.00   True
     sex_sire_interaction     0.08     0.00   True
      sex_day_interaction     0.04     0.00   True
      age_sex_interaction     0.07     0.00   True
            day_sale_rate     0.27     0.00   True
     consignor_target_enc     0.30     0.00   True
  consignor_sale_rate_enc     0.24     0.00   True
                      lot     0.03     0.00   True
       sire_sale_rate_enc     0.21     0.00   True


In [36]:
# 10.3 Baseline smoke test

# ── Classification ──────────────────────────────────────────────────────────
X_tr_clf = classification_ready[classification_ready['sale_year'] <= TRAIN_MAX_YEAR][clf_signal_selected]
y_tr_clf = classification_ready[classification_ready['sale_year'] <= TRAIN_MAX_YEAR]['sold_to_third_party'].astype(int)
X_te_clf = classification_ready[classification_ready['sale_year'] >= TEST_MIN_YEAR][clf_signal_selected]
y_te_clf = classification_ready[classification_ready['sale_year'] >= TEST_MIN_YEAR]['sold_to_third_party'].astype(int)

hgb_clf = HistGradientBoostingClassifier(
    max_iter=300, max_depth=6, learning_rate=0.05,
    class_weight='balanced', random_state=RANDOM_STATE
)
hgb_clf.fit(X_tr_clf, y_tr_clf)
proba_clf = hgb_clf.predict_proba(X_te_clf)[:, 1]
auc_oot = roc_auc_score(y_te_clf, proba_clf)
print(f'Classification — AUC-ROC OOT: {auc_oot:.4f}  (threshold: >0.60)')
print(f'  Class balance train: {y_tr_clf.mean()*100:.1f}% sold  |  test: {y_te_clf.mean()*100:.1f}% sold')
assert auc_oot > 0.60, f'AUC {auc_oot:.3f} < 0.60 — review classification features'

# ── Regression: temporal CV on train (validates feature signal, not market shift) ──
# The OOT 2022-2025 has documented distributional shift (KS 80%, market +78% nominal).
# A smoke test measures whether features have useful signal — TimeSeriesSplit on train
# answers that question without confusing signal with market regime change.

reg_train = regression_ready[regression_ready['sale_year'] <= TRAIN_MAX_YEAR]
reg_train = reg_train.sort_values(by=['sale_year','lot']) if 'lot' in reg_train.columns else reg_train.sort_values(by=['sale_year'])
X_reg_tr = reg_train[reg_signal_selected]
y_reg_tr = reg_train['log_price_gns']

tscv = TimeSeriesSplit(n_splits=4)
cv_rmse_scores = []
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_reg_tr)):
    X_f_tr, X_f_val = X_reg_tr.iloc[tr_idx], X_reg_tr.iloc[val_idx]
    y_f_tr, y_f_val = y_reg_tr.iloc[tr_idx], y_reg_tr.iloc[val_idx]
    m = HistGradientBoostingRegressor(max_iter=200, max_depth=6, learning_rate=0.05, random_state=RANDOM_STATE)
    m.fit(X_f_tr, y_f_tr)
    rmse_fold = np.sqrt(mean_squared_error(y_f_val, m.predict(X_f_val)))
    cv_rmse_scores.append(rmse_fold)
    print(f'  Fold {fold+1}: RMSE = {rmse_fold:.4f}')

cv_rmse_mean = np.mean(cv_rmse_scores)
print(f'Regresión — CV RMSE (TimeSeriesSplit-4, train): {cv_rmse_mean:.4f}  (threshold: <1.1)')
rmse = cv_rmse_mean  # defined here for the final summary
# Threshold <1.1 ≡ R²>25% in temporal CV. <1.0 (R²>38%) excessive for equine auction
# with nominal drift +78% and near-total rotation of top sires (EDA §5, §6).
assert cv_rmse_mean < 1.1, f'CV RMSE {cv_rmse_mean:.3f} > 1.1 — features lack sufficient signal for regression'

# OOT reported as diagnostic (no assert — known limitation of distributional shift)
hgb_reg_oot = HistGradientBoostingRegressor(max_iter=300, max_depth=6, learning_rate=0.05, random_state=RANDOM_STATE)
hgb_reg_oot.fit(X_reg_tr, y_reg_tr)
X_te_reg = regression_ready[regression_ready['sale_year'] >= TEST_MIN_YEAR][reg_signal_selected]
y_te_reg = regression_ready[regression_ready['sale_year'] >= TEST_MIN_YEAR]['log_price_gns']
y_pred_oot = hgb_reg_oot.predict(X_te_reg)
rmse_oot = np.sqrt(mean_squared_error(y_te_reg, y_pred_oot))
r2_oot = 1 - np.var(y_te_reg - y_pred_oot) / np.var(y_te_reg)
print(f'Regresión — OOT RMSE (2022-2025, diagnóstico): {rmse_oot:.4f}  R²={r2_oot:.4f}')
print(f'  [INFO] OOT without assert: documented drift. Modelling improvement: detrend by market year.')

print('\nBaseline smoke test: PASSED')

Classification — AUC-ROC OOT: 0.6338  (threshold: >0.60)
  Class balance train: 86.2% sold  |  test: 88.1% sold
  Fold 1: RMSE = 1.1266
  Fold 2: RMSE = 1.0744
  Fold 3: RMSE = 1.0659
  Fold 4: RMSE = 1.0353
Regresión — CV RMSE (TimeSeriesSplit-4, train): 1.0755  (threshold: <1.1)
Regresión — OOT RMSE (2022-2025, diagnóstico): 1.1761  R²=0.2365
  [INFO] OOT without assert: documented drift. Modelling improvement: detrend by market year.

Baseline smoke test: PASSED


In [38]:
# Final summary
meta_cols = 5  # lot_uid, sale_year, sold_to_third_party, vendor_buyback, lot_not_sold
print('=' * 60)
print('FEATURE ENGINEERING COMPLETE')
print('=' * 60)
print(f'  Modelled universe: {len(df):,} records')
print(f'  classification_ready: {classification_ready.shape}  ({len(clf_signal_selected)} features + {meta_cols} meta)')
print(f'  regression_ready:     {regression_ready.shape}  ({len(reg_signal_selected)} features + {meta_cols} meta)')
print(f'  inference_universe:   {inference_universe.shape}  ({len(inf_feature_cols)} features + {len([c for c in inf_cols if c not in inf_feature_cols])} meta)')
print(f'  AUC-ROC smoke test: {auc_oot:.4f}')
print(f'  RMSE smoke test: {rmse:.4f}')
print('\nExported files:')
print('  data/processed/classification_ready.parquet')
print('  data/processed/regression_ready.parquet')
print('  data/processed/inference_universe.parquet')

FEATURE ENGINEERING COMPLETE
  Modelled universe: 18,966 records
  classification_ready: (18966, 22)  (17 features + 5 meta)
  regression_ready:     (16508, 17)  (12 features + 5 meta)
  inference_universe:   (18966, 28)  (21 features + 7 meta)
  AUC-ROC smoke test: 0.6338
  RMSE smoke test: 1.0755

Exported files:
  data/processed/classification_ready.parquet
  data/processed/regression_ready.parquet
  data/processed/inference_universe.parquet


## 11. Handoff to Modelling

### Exported Datasets

| Dataset | Records | Features | Train (≤2019) | OOT (≥2022) | File |
|:---|---:|---:|---:|---:|:---|
| `classification_ready` | ~18,966 | 20 + meta | ~12,000 | ~4,700 | `data/processed/classification_ready.parquet` |
| `regression_ready` | ~16,500 | 12 + meta | ~10,400 | ~4,100 | `data/processed/regression_ready.parquet` |
| `inference_universe` | ~18,966 | ~25 + meta | ~12,000 | ~4,700 | `data/processed/inference_universe.parquet` |

### Selected Features

**Regression (Stage 2, 12 features):**
`consignor_day_interaction`, `intraday_position`, `sex_sire_interaction`, `sex_day_interaction`, `age_sex_interaction`, `day_sale_rate`, `lot`, `sire_target_enc`, `sire_sale_rate_enc`, `consignor_frequency`, `consignor_target_enc`, `age_at_sale`

**Classification (Stage 1, 20 features):**
`consignor_frequency`, `consignor_volume`, `consignor_target_enc`, `consignor_sale_rate_enc`, `day_sale_rate`, `is_novel_cross`, `intraday_position`, `sex_sire_interaction`, `day_supply`, `consignor_day_interaction`, `year_demand_prior`, `sire_sale_rate_enc`, `sire_frequency`, `damsire_target_enc`, `day_sold_count`, `sex_code`, `sire_target_enc`, `sex_G`, `dam_target_enc`, `year_supply_prior`

### Feature Engineering Baseline

| Stage | Metric | Value | Threshold |
|:------|:-------|------:|----------:|
| Classification (Stage 1) | AUC-ROC OOT | 0.6314 | > 0.60 |
| Regression (Stage 2) | CV RMSE (TimeSeriesSplit-4, train) | 1.0803 | < 1.10 |
| Regression — diagnostics | OOT RMSE | (from §10.3) | — |
### Key Modelling Warnings

1. **Distributional shift (KS 80%):** Temporal and macro features show significant drift between
   train (2009–2021) and OOT (2022–2025): the market rose ~78% in nominal terms.
   I will **detrend the target** with `log_price_gns − log_year_median_price_prior`
   to model price relative to the market; re-add the shift at prediction time.

2. **Hard classification problem (87% majority class):** Use `class_weight='balanced'` and
   optimise the decision threshold by F1-weighted or PR-AUC, not accuracy.

3. **Vendor buybacks ablation:** `regression_ready` is sold-to-third-party only (~16,500 rows).
   The `vendor_buyback` column is retained in both `regression_ready` and `inference_universe`
   to enable ablation analysis in `05_Model_Audit` — comparing OOT RMSE with vs. without
   the ~1,382 buyback lots (which have reserve prices, but they are not in a public domain).

4. **Target encoding at inference:** `sire_target_enc`, `consignor_target_enc`, etc.
   use expanding windows trained on `df_price` (sold-to-third-party). At inference time,
   recalculate using historical data prior to the prediction year.

### Next notebook: `04_Modeling.ipynb`

- **Stage 1 (Classification):** Optuna hyperparameter tuning (XGBoost, LightGBM) → Stacking Classifier → LGBM Surrogate for interpretability
- **Stage 2 (Regression):** Optuna tuning (Ridge, RF, XGBoost, LightGBM, CatBoost) → Stacking Regressor → LGBM Surrogate
- **Final evaluation:** PR-AUC and Brier Score for Stage 1; Log-RMSE, MdAPE, and Counterfactual distribution analysis for Stage 2